In [1]:
# Funcao minima benchmark
from time import perf_counter
from statistics import median
import tracemalloc
import numpy as np

def benchmark(func, *args, repeat=9):
    tempos = []
    resultado = None
    for _ in range(repeat):
        t0 = perf_counter()
        resultado = func(*args)
        tempos.append(perf_counter() - t0)
    return resultado, median(tempos), tempos

In [2]:
# Funcao test
def matrix_sum(A):
    sum = 0
    for i in range(len(A)):
        for j in range(len(A[i])):
            sum += A[i][j]
    return sum

rows = 1000
cols = 1000
A = np.random.rand(rows, cols)

In [3]:
benchmark(matrix_sum, A) 

(np.float64(499883.91492922953),
 0.14038533399980224,
 [0.19877065700029561,
  0.1817547629998444,
  0.16500162800002727,
  0.15103276400031973,
  0.14038533399980224,
  0.132379077999758,
  0.1284260180000274,
  0.11843953600009627,
  0.11491884900033256])

In [4]:
# Versao A: recalcula uma constante n vezes 
def versao_a(x, a, b):
    s = 0.0 
    for xi in x:
        s += xi * ((a**2 + b**2) ** 0.5)
    return s 

In [5]:
# Versao B: hoisting - move o invariante pra fora do laço 
def versao_b(x, a, b):
    c = (a**2 + b**2) ** 0.5
    s = 0.0
    for xi in x:
        s += xi * c 
    return s

In [6]:
a = 10 
b = 100 
x = np.random.rand(1000000)

In [7]:
benchmark(versao_a, x, a, b)

(np.float64(50207859.075159915),
 0.14345311699980812,
 [0.15293298899996444,
  0.14390078999986144,
  0.14557255599993368,
  0.142223037000349,
  0.1427812840001934,
  0.14189357599980212,
  0.14345311699980812,
  0.1461067049999656,
  0.1419239869997])

In [8]:
benchmark(versao_b, x, a, b)

(np.float64(50207859.075159915),
 0.08393898399981481,
 [0.08393898399981481,
  0.08322770099994159,
  0.08305287499979386,
  0.08675533799987534,
  0.08619005900027332,
  0.08403387799990014,
  0.08457673600014459,
  0.07807447000004686,
  0.08012813799996366])

In [9]:
# Estrutura de dados altera o custo da busca
import random

dados = list(range(100_000))
consultas = [random.randrange(120_000) for _ in range(5_000)]

def busca_lista(dados, consultas):
    return sum(q in dados for q in consultas)

def busca_set(dados, consultas):
    indice = set(dados)
    return sum(q in indice for q in consultas)

In [10]:
benchmark(busca_lista, dados, consultas)

(4179,
 1.308513646999927,
 [1.3341772130002028,
  1.3179235570000856,
  1.3208805849999408,
  1.3127216739999312,
  1.308513646999927,
  1.3028086969998185,
  1.3045635940002285,
  1.3015191650001725,
  1.2996186020000096])

In [11]:
benchmark(busca_set, dados, consultas)

(4179,
 0.0013331039999684435,
 [0.004480949000026158,
  0.004441927999778272,
  0.0012366710002424952,
  0.0011994199999207922,
  0.0012195409999549156,
  0.0012819829998989007,
  0.0013331039999684435,
  0.0014830099999016966,
  0.001368575000014971])

In [12]:
# 4 - microbenchmarkd de overhead de chamada
def quadrado(x):
    return x * x 

def com_chamada(n):
    s = 0 
    for i in range(n):
        s += quadrado(i)
    return s 

def inline(n):
    s = 0 
    for i in range(n):
        s += i*i 
    return s

In [13]:
benchmark(com_chamada, 100000)

(333328333350000,
 0.004805340000075375,
 [0.004626863999874331,
  0.005158831999779068,
  0.004736088000299787,
  0.004805340000075375,
  0.004629324000234192,
  0.00460510399989289,
  0.005285976000322989,
  0.007927053999992495,
  0.004916013999718416])

In [14]:
benchmark(inline, 100000)

(333328333350000,
 0.0032937799996943795,
 [0.003254208000271319,
  0.0032937799996943795,
  0.0032410680000793946,
  0.003521706999890739,
  0.0033997430000454187,
  0.003250637999826722,
  0.0032994209996104473,
  0.003815707999820006,
  0.0032040470000538335])

In [26]:
# pre alocacao 
def benchmark_memo(func, *args, repeat=9):
    memos = []
    resultado = None
    for _ in range(repeat):
        tracemalloc.start()
        resultado = func(*args)
        current, peak = tracemalloc.get_traced_memory()
        memos.append(current)
        tracemalloc.stop()
    return resultado, median(memos), memos 


def preencher_lista(n):
    y = []
    for i in range(n):
        y.append(i * i)
    return y 

def preencher_array(n):
    y = np.empty(n, dtype=np.int64)
    for i in range(n):
        y[i] = i * i 
    return y

In [21]:
rst, md, tmp = benchmark(preencher_lista, 100_000);
md, tmp

(0.00399165399994672,
 [0.0037440750002133427,
  0.004364025999620935,
  0.003975182999965909,
  0.004010694000044168,
  0.003873938999731763,
  0.004202691000045888,
  0.004301784000290354,
  0.00399165399994672,
  0.003877909000038926])

In [22]:
benchmark(preencher_array, 100_000)

(array([         0,          1,          4, ..., 9999400009, 9999600004,
        9999800001]),
 0.008156251999935193,
 [0.009610084999621904,
  0.008138852999763913,
  0.010740044999693055,
  0.00815646299997752,
  0.00849230499989062,
  0.008019848000003549,
  0.008156251999935193,
  0.0076770050000050105,
  0.007829471999684756])

In [27]:
rst, md, tmp = benchmark_memo(preencher_lista, 100_000);
md, tmp

(4000384,
 [4001484,
  4000384,
  4000384,
  4000384,
  4000384,
  4000384,
  4000384,
  4000384,
  4000384])

In [28]:
benchmark_memo(preencher_array, 100_000)

(array([         0,          1,          4, ..., 9999400009, 9999600004,
        9999800001]),
 800096,
 [801171, 800096, 800096, 800096, 800096, 800096, 800096, 800096, 800096])